In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import joblib
import re
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import sys

from sklearn.model_selection import train_test_split
from plotly.subplots import make_subplots
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.feature_extraction.text import CountVectorizer   # Sac de mots
from sklearn.feature_extraction.text import TfidfVectorizer    # TF-IDF
from sklearn.feature_extraction.text import (
    ENGLISH_STOP_WORDS  #Stop words English
)
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import LinearSVC
import pickle
import nltk
import shutil
import os
import glob
import warnings

import optuna

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, KFold, cross_val_score, RandomizedSearchCV, cross_val_predict
from sklearn.metrics import (
    accuracy_score,  # Précision globale
    precision_score,  # Précision
    recall_score,  # Rappel
    f1_score,  # F1-Score
    confusion_matrix,  # Matrice de confusion
    roc_auc_score,  # AUC (Area Under the Curve)
    roc_curve,  # Courbe ROC
    classification_report,  # Rapport de classification
    mean_squared_error,  # Erreur quadratique moyenne (MSE)
    mean_absolute_error,  # Erreur absolue moyenne (MAE)
    r2_score,  # Coefficient de détermination R²
    auc  # Calcul de l'AUC
)
from sklearn.exceptions import ConvergenceWarning

# Librairies pour LDA
import gensim                    # Modélisation des topics et NLP
from gensim import corpora       # Création du dictionnaire de mots
from gensim.models import LdaModel  # Modèle LDA simple
from gensim.utils import simple_preprocess  # Prétraitement simple des textes
from gensim.models import CoherenceModel    # Calcul de la cohérence des topics
from gensim.models import Phrases           # Création de bigrammes/trigrammes
from gensim.models.phrases import Phraser   # Optimisation bigrammes/trigrammes
from gensim import corpora, models          # Utilisation des modèles
from gensim.models import LdaMulticore      # Modèle LDA parallèle
import multiprocessing                      # Gestion du multi-threading


nltk_data_dir = './nltk_data'


# On laisse au cas où
# # Aggressively clean corrupted NLTK data
# if os.path.exists(nltk_data_dir):
#     shutil.rmtree(nltk_data_dir)
# # Also remove any corrupted zip files from system paths
# for path in nltk.data.path:
#     if os.path.exists(path):
#         for zipfile in glob.glob(os.path.join(path, '*.zip')):
#             try:
#                 os.remove(zipfile)
#             except:
#                 pass

# Create fresh nltk_data directory
os.makedirs(nltk_data_dir, exist_ok=True)
nltk.data.path.insert(0, nltk_data_dir)

# Download resources with fresh start
print("Downloading NLTK resources...")
nltk.download('wordnet', download_dir=nltk_data_dir, quiet=True)
nltk.download('omw-1.4', download_dir=nltk_data_dir, quiet=True)
nltk.download('wordnet_ic', download_dir=nltk_data_dir, quiet=True)
nltk.download('averaged_perceptron_tagger', download_dir=nltk_data_dir, quiet=True)
print("NLTK resources downloaded successfully!")

from nltk.corpus import wordnet
from nltk.stem.snowball import SnowballStemmer

from contraction_fix import fix as expand_contractions # Contraction

In [ ]:
import spacy                                # NLP avancé
from spacy import displacy                  # Visualisation
from spacy.cli import download
download("en_core_web_sm")

In [ ]:
nlp = spacy.load("en_core_web_sm")

In [ ]:
def stop_words(text):
    # Normalisation simple des apostrophes typographiques
    text = text.replace("’", "'")
    tokens = text.split()
    expanded = []
    for t in tokens:
        key = t.lower()
        if key not in ENGLISH_STOP_WORDS:
            expanded.append(t)
    return " ".join(expanded)

text = "I'm happy but I don't know why."
print(text)
print(stop_words(expand_contractions(text)))

In [ ]:
stemmer = nltk.stem.porter.PorterStemmer()
lemmatizer = nltk.stem.WordNetLemmatizer()

def stemmerLemma(text):
    # Normalisation simple des apostrophes typographiques
    text = text.replace("’", "'")
    expanded = []
    t2 = nlp(text)
    
    for t in t2:
        
        key = t.text.lower()
        # print(key, t.pos_, stemmer.stem(key), lemmatizer.lemmatize(key))
        if t.pos_ == "VERB":
            result = stemmer.stem(key)
        elif t.pos_ == "NOUN" or t.pos_ == "ADJ" or t.pos_ == "PROPN":
            result = lemmatizer.lemmatize(key)
        else:
            result = key
        expanded.append(result)
    return " ".join(expanded)

In [ ]:
emoticons_str = r"""
    (?:
        [:=;] # Eyes
        [oO\-]? # Nose (optional)
        [D\)\]\(\]/\\OpP] # Mouth
    )"""
html_str = r"<[^>]+>"
mentions_str = r"(?:@[\w_]+)"
hashtags_str = r"(?:\#+[\w_]+[\w\'_\-]*[\w_]+)"
url_str = r"http[s]?://(?:[a-z]|[0-9]|[$-_@.&amp;+]|[!*\(\),]|(?:%[0-9a-f][0-9a-f]))+"
number_str = r"(?:(?:\d+,?)+(?:\.?\d+)?)"
compose_str = r"(?:[a-z][a-z'\-_]+[a-z])"
mots_str = r"(?:[\w_]+)"
reste_str = r"(?:[\S]+)"

total_str = [
    emoticons_str,
    html_str,
    mentions_str,
    hashtags_str,
    url_str,
    number_str,
    compose_str,
    mots_str,
    reste_str,
]

emoticons_regex = re.compile(emoticons_str, re.VERBOSE | re.IGNORECASE)
html_regex = re.compile(html_str, re.VERBOSE | re.IGNORECASE)
mentions_regex = re.compile(mentions_str, re.VERBOSE | re.IGNORECASE)
hashtags_regex = re.compile(hashtags_str, re.VERBOSE | re.IGNORECASE)
url_regex = re.compile(url_str, re.VERBOSE | re.IGNORECASE)
number_regex = re.compile(number_str, re.VERBOSE | re.IGNORECASE)
compose_regex = re.compile(compose_str, re.VERBOSE | re.IGNORECASE)
mots_regex = re.compile(mots_str, re.VERBOSE | re.IGNORECASE)
reste_regex = re.compile(reste_str, re.VERBOSE | re.IGNORECASE)
total_regex = re.compile(r'('+'|'.join(total_str)+')', re.VERBOSE | re.IGNORECASE)

def preprocess(s, emoticons=False, html=False, mentions=False, hashtags=False, url=False, number=False, compose=True,
               stemmerlemma=True, stopwords=True):
    tokens = total_regex.findall(s)

    tokens = [tok for tok in tokens if 
        mots_regex.search(tok) and # TODO: fix
        (emoticons or not emoticons_regex.search(tok)) and
        (html or not html_regex.search(tok)) and
        (mentions or not mentions_regex.search(tok)) and
        (hashtags or not hashtags_regex.search(tok)) and
        (url or not url_regex.search(tok)) and
        (number or not number_regex.search(tok)) and
        (compose or not compose_regex.search(tok))
    ]
            
    sentence = " ".join(tokens)
    if stemmerlemma:
        sentence = stemmerLemma(sentence)
    if stopwords:
        sentence = stop_words(sentence)
        
    return sentence

preprocess("This is an example, ! of #CountVectorizer for creating a vector https://leotta.ro 런쥔을공평하게_대하세요")
# print(preprocess("This is another example of CountVectorizer"))
# print(preprocess("with or without parameters"))

# Répartition des données (upsampling / downsampling / rien)

In [ ]:
df_lue=pd.read_csv('scitweets_export.tsv',
sep='\t')
# print (df.head())
# print (df.shape)
# print (df.columns)

# sampling="up"
# sampling="down"
sampling="nothing"

# ---------------------------------------------------------------
# ---------------------------------------------------------------
# ---------------------------------------------------------------

# 1. On sépare les classes du df d'origine
df_min = df_lue[df_lue["science_related"] == 0]
df_max = df_lue[df_lue["science_related"] == 1]
if len(df_min) > len(df_max):
    df_min,df_max=df_max,df_min
nb_lignes_min = len(df_min)
nb_lignes_max = len(df_max)


if sampling=="up":
    df_min = df_min.sample(n=nb_lignes_max, replace=True, random_state=42)
elif sampling=="down":
    df_max = df_max.sample(n=nb_lignes_min, random_state=42)

df = pd.concat([df_min, df_max])

# si upsampling on boost le nombre d'échantillons du df
# if sampling=="up": 
#     taille_finale = int(len(df) * 1.5) # augmentation de 50% de la taille
#     df = df.sample(n=taille_finale, replace=True, random_state=42) # 2. On duplique l'ensemble du dataset équilibré

# On mélange le tout pour que ça soit propre (pas de grande suite de 0 ou de 1)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# --- VÉRIFICATION ---
print(f"Taille d'origine : {len(df_lue)} lignes")
print(f"Taille finale : {len(df)} lignes")
print("\nRépartition finale des classes :")
print(df["science_related"].value_counts())

# Benchmark des models

In [ ]:
def plot_confusion_matrix(cm, classes, title='Matrice de confusion', cmap=plt.cm.Blues):
    """
    Affiche la matrice de confusion.
    
    Parameters:
    - cm (array-like): Matrice de confusion (2D numpy array).
    - classes (list of str): Liste des noms des classes correspondant aux dimensions de la matrice.
    - title (str): Titre du graphique (par défaut 'Matrice de confusion').
    - cmap (matplotlib.colors.Colormap): Carte des couleurs à utiliser pour le graphique (par défaut plt.cm.Blues).
    """
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, xticklabels=classes, yticklabels=classes)
    plt.title(title)
    plt.ylabel('Vérité terrain')
    plt.xlabel('Prédictions')
    plt.tight_layout()
    plt.show()

In [ ]:
texte = [preprocess(t) for t in df.text]
pipe = Pipeline([
    ("vectorizer", TfidfVectorizer(lowercase=False, ngram_range=(1, 2))),
    #("SVM", SVC(kernel="linear"))
    ("model", LinearSVC())
])

X_train_related, X_test_related, y_train_related, y_test_related = train_test_split(
    texte, df["science_related"], test_size=0.2, stratify=df["science_related"], random_state=42
)

In [ ]:
# 1. Préparation de tes données textuelles brutes
X_raw = [preprocess(t) for t in df["text"]]
y = df["science_related"]

# 2. Liste de tes modèles (sans GaussianNB, remplacé par MultinomialNB)
models = [
    ("KNN", KNeighborsClassifier()),
    ("DecisionTree", DecisionTreeClassifier()),
    ("MultinomialNB", MultinomialNB()),
    ("LinearSVC", LinearSVC()),
    ("RandomForest", RandomForestClassifier())
]

metriques = ["accuracy", "recall", "precision", "f1"]

names = []
results = {m:[] for m in metriques}

seed = 42
kfold = KFold(n_splits=10, random_state=seed, shuffle=True)

print("Début de l'évaluation des modèles...\n")

for name, model in models:
    names.append(name)
    print(f"{"-"*15} Modèle : {name} {"-"*15}")
    
    # ÉTAPE CLÉ : Le Pipeline. 
    # À chaque tour de validation croisée, il va vectoriser UNIQUEMENT sur le train set,
    # puis prédire sur le test set. Zéro fuite de données !
    pipeline = Pipeline([
        ("vectorizer", TfidfVectorizer(lowercase=False, ngram_range=(1, 2))),
        ("model", model)
    ])
    
    # Génération des prédictions "out-of-fold" pour générer les rapports
    y_pred = cross_val_predict(pipeline, X_raw, y, cv=kfold)
    
    print("\nClassification report :")
    print(classification_report(y, y_pred))
    print("\n\n")
    
    conf_matrix_balanced = confusion_matrix(y, y_pred)
    labels = ["non-scientifique", "scientifique"]
    plot_confusion_matrix(conf_matrix_balanced, labels, title=("Matrice de confusion de " + name))



    # Calcul de l'accuracy moyenne pour ce modèle
    for metrique in metriques:
        cv_results = cross_val_score(pipeline, X_raw, y, cv=kfold, scoring=metrique)
        results[metrique].append(cv_results)

In [ ]:
for metrique in metriques:
    fig = plt.figure()
    fig.suptitle("Comparaison des " + metrique)
    ax = fig.add_subplot(111)
    plt.boxplot(results[metrique])
    ax.set_xticklabels(names)
    plt.show()

# Creation du big model oulala

# chez pas quoi

In [ ]:
# --- 1. Création de la Pipeline ---
pipe = Pipeline([
    ("vectorizer", TfidfVectorizer(lowercase=False)),
    ("model", RandomForestClassifier(random_state=42))
])

# --- 2. Définition des hyperparamètres ---
# On fusionne en un seul dictionnaire car le Random Forest n'a pas de conflits de paramètres comme le LinearSVC (l1 vs l2)
params = {
    "model__n_estimators": [4, 6, 9],
    "model__max_features": ["log2", "sqrt"],
    "model__criterion": ["entropy", "gini"],
    "model__max_depth": [2, 3, 5, 10],
    "model__min_samples_split": [2, 3, 5],
    "model__min_samples_leaf": [1, 5, 8],
    
    "vectorizer__ngram_range": [(1,1), (1,2), (1,3)],
    "vectorizer__max_features": [3000, 5000, 10000],
    "vectorizer__min_df": [2, 5, 10],
    "vectorizer__max_df": [0.85, 0.95],
}

# --- 3. Configuration de la validation croisée ---
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
fig = plt.figure()
fig.suptitle("Comparaison des algorithmes")
ax = fig.add_subplot(111)
plt.boxplot(results)
ax.set_xticklabels(names)
plt.show()
grid = GridSearchCV(
    pipe, 
    params,
    cv=cv,
    scoring="f1_weighted",
    return_train_score=True,
    n_jobs=-1
)

# --- 4. Entraînement ---
# Plus besoin du filtre ConvergenceWarning, il n'est pas utilisé par Random Forest
grid.fit(X_train_related, y_train_related)

# --- 5. Diagnostic d'Overfitting ---
results = pd.DataFrame(grid.cv_results_)
train_score = results.loc[grid.best_index_, "mean_train_score"]
val_score   = grid.best_score_
gap         = train_score - val_score

print("Meilleurs paramètres :", grid.best_params_)
print(f"Train score:      {train_score:.4f}")
print(f"Validation score: {val_score:.4f}")
print(f"Overfitting gap:  {gap:.4f}", "⚠️ possible overfit" if gap > 0.1 else "✅ looks healthy")

# --- 6. Évaluation Finale ---
test_score = grid.score(X_test_related, y_test_related)
print(f"\nTest score (held-out): {test_score:.4f}")
print(classification_report(y_test_related, grid.predict(X_test_related)))

# La précision mesure la proportion de prédictions positives correctes parmi toutes les prédictions positives faites par le modèle

In [ ]:
# --- 1. Création de la Pipeline ---
pipe = Pipeline(
    [
        ("vectorizer", TfidfVectorizer(lowercase=False)),
        (
            "model",
            RandomForestClassifier(random_state=42),
        ),  # <--- CHANGEMENT DU MODÈLE ICI
    ]
)

# (Je pars du principe que X_train_related, etc. sont déjà créés avec votre train_test_split original)

# --- 2. Définition des hyperparamètres ---
# On fusionne en un seul dictionnaire car le Random Forest n'a pas de conflits de paramètres comme le LinearSVC (l1 vs l2)
params = {
    # Paramètres spécifiques au Random Forest
    "model__n_estimators": [4, 6, 9],
    "model__max_features": ["log2", "sqrt"],
    "model__criterion": ["entropy", "gini"],
    "model__max_depth": [2, 3, 5, 10, 15, 20],
    "model__min_samples_split": [2, 3, 5],
    "model__min_samples_leaf": [1, 5, 8],
    # Paramètres du TfidfVectorizer (conservés à l'identique)
    "vectorizer__ngram_range": [(1, 1), (1, 2), (1, 3)],
    "vectorizer__max_features": [3000, 5000, 10000],
    "vectorizer__min_df": [2, 5, 10],
    "vectorizer__max_df": [0.85, 0.95],
}

# --- 3. Configuration de la validation croisée ---
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

grid = RandomizedSearchCV(
    pipe,
    params,
    n_iter=1000,   # au lieu de 23328 combinaisons
    cv=cv,
    scoring="f1_weighted",
    n_jobs=-1,
    random_state=42,
    return_train_score=True
)


# --- 4. Entraînement ---
# Plus besoin du filtre ConvergenceWarning, il n'est pas utilisé par Random Forest
grid.fit(X_train_related, y_train_related)

# --- 5. Diagnostic d'Overfitting ---
results = pd.DataFrame(grid.cv_results_)
train_score = results.loc[grid.best_index_, "mean_train_score"]
val_score = grid.best_score_
gap = train_score - val_score

print("Meilleurs paramètres :", grid.best_params_)
print(f"Train score:      {train_score:.4f}")
print(f"Validation score: {val_score:.4f}")
print(
    f"Overfitting gap:  {gap:.4f}",
    "⚠️ possible overfit" if gap > 0.1 else "✅ looks healthy",
)

# --- 6. Évaluation Finale ---
test_score = grid.score(X_test_related, y_test_related)
print(f"\nTest score (held-out): {test_score:.4f}")
print(classification_report(y_test_related, grid.predict(X_test_related)))

# La précision mesure la proportion de prédictions positives correctes parmi toutes les prédictions positives faites par le modèle

In [ ]:
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV
from joblib import Memory

# import pandas as pd

# =========================================================
# 1. Cache du pipeline (accélère fortement le TF-IDF)
# =========================================================

memory = Memory("./cache_dir", verbose=0)

# =========================================================
# 2. Pipeline
# =========================================================

pipe = Pipeline(
    [
        ("vectorizer", TfidfVectorizer(lowercase=False)),
        ("model", RandomForestClassifier(random_state=42, n_jobs=-1)),
    ],
    memory=memory,
)

# =========================================================
# 3. Hyperparamètres
# (version réduite et plus réaliste)
# =========================================================

params = {
    # Random Forest
    "model__n_estimators": [50, 100],
    "model__max_features": ["sqrt"],
    "model__criterion": ["gini", "entropy"],
    "model__max_depth": [5, 10, None],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2],
    # TF-IDF
    "vectorizer__ngram_range": [(1, 1), (1, 2)],
    "vectorizer__max_features": [3000, 5000],
    "vectorizer__min_df": [2, 5],
    "vectorizer__max_df": [0.85, 0.95],
}

# =========================================================
# 4. Validation croisée
# =========================================================

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# =========================================================
# 5. Halving Grid Search
# =========================================================

grid = HalvingGridSearchCV(
    estimator=pipe,
    param_grid=params,
    # métrique
    scoring="f1_weighted",
    # CV
    cv=cv,
    # parallélisation
    n_jobs=-1,
    # stratégie halving
    factor=3,  # élimine ~2/3 des modèles à chaque étape
    resource="n_samples",
    max_resources="auto",
    # diagnostics
    verbose=2,
    # reproductibilité
    random_state=42,
)

# =========================================================
# 6. Entraînement
# =========================================================

grid.fit(X_train_related, y_train_related)

# =========================================================
# 7. Résultats
# =========================================================

results = pd.DataFrame(grid.cv_results_)

train_score = results.loc[grid.best_index_, "mean_train_score"]
val_score = grid.best_score_
gap = train_score - val_score

print("\n==============================")
print("Meilleurs paramètres")
print("==============================")
print(grid.best_params_)

print(f"\nTrain score:      {train_score:.4f}")
print(f"Validation score: {val_score:.4f}")
print(
    f"Overfitting gap:  {gap:.4f}",
    "⚠️ possible overfit" if gap > 0.1 else "✅ looks healthy",
)

# =========================================================
# =========================================================

test_score = grid.score(X_test_related, y_test_related)

print(f"\nTest score (held-out): {test_score:.4f}")

y_pred = grid.predict(X_test_related)

print("\nClassification Report")
print(classification_report(y_test_related, y_pred))

In [ ]:
model_path = "ScientificModelTree.pkl"
with open(model_path, "wb") as f:
    pickle.dump(grid.best_estimator_, f)
print("Modèle sauvegardé dans :", model_path)

L'intuition du Recall 😗 il mesure la proportion de cas positifs correctement identifiés parmi tous les cas réellement positifs. Il répond donc à la question :parmi tous les individus qui sont réellement malades, combien ont été correctement identifiés par le modèle ?*

In [ ]:
def objective_linear(trial):
    ngram_max = trial.suggest_int("ngram_max", 1, 3)
    param_ngram_range = (1, ngram_max)
    param_max_features = trial.suggest_int("max_features", 3000, 10000, step=500)

    param_min_df = trial.suggest_int("min_df", 2, 10)

    param_max_df = trial.suggest_float("max_df", 0.70, 0.99)

    param_C = trial.suggest_float("C", 1e-3, 1e2, log=True)

    param_max_iter = trial.suggest_int("max_iter", 5000, 20000, step=1000)

    param_penalty = trial.suggest_categorical("penalty", ["l1", "l2"])

    if param_penalty == "l1":
        param_loss = "squared_hinge"
        param_dual = False
    else:
        param_loss = trial.suggest_categorical("loss", ["hinge", "squared_hinge"])
        param_dual = True if param_loss == "hinge" else "auto"

    if param_loss == "hinge":
        param_dual = True
    else:
        param_dual = "auto"

    vectorizer = TfidfVectorizer(
        ngram_range=param_ngram_range,
        max_features=param_max_features,
        min_df=param_min_df,
        max_df=param_max_df,
        lowercase=False,
    )

    model = LinearSVC(
        C=param_C,
        penalty=param_penalty,
        loss=param_loss,
        max_iter=param_max_iter,
        dual=param_dual,
        random_state=42,
    )

    pipeline = Pipeline([("vectorizer", vectorizer), ("model", model)])

    # ---------------------------------------------------------
    # 4. ÉVALUATION (CROSS-VALIDATION)
    # ---------------------------------------------------------
    kfold = KFold(n_splits=5, shuffle=True, random_state=42)

    scores = cross_val_score(
        pipeline, X_train_related, y_train_related, cv=kfold, scoring="accuracy", n_jobs=-1
    )

    return scores.mean()


# --- Lancement de l'optimisation ---
study = optuna.create_study(direction="maximize")
study.optimize(objective_linear, n_trials=30)

print("\nMeilleurs paramètres :", study.best_params)
print("Meilleur score :", study.best_value)

In [ ]:
def objective_forest(trial):
    n_estimators = trial.suggest_int("n_estimators", 10, 200)
    max_depth = trial.suggest_int("max_depth", 10, 20)
    max_features = trial.suggest_categorical("max_features", ["sqrt", "log2"])
    criterion = trial.suggest_categorical("criterion", ["gini", "entropy"])
    min_samples_split = trial.suggest_int("min_samples_split", 2, 10)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10)

    clf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        max_features=max_features,
        criterion=criterion,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        random_state=0,
    )

    # Use a pipeline so the raw text is vectorized before fitting the classifier
    pipe = Pipeline([
        ("vectorizer", TfidfVectorizer(lowercase=False, ngram_range=(1, 2))),
        ("clf", clf),
    ])

    kfold = KFold(n_splits=5, shuffle=True, random_state=0)

    scores = cross_val_score(
        pipe, X_train_related, y_train_related, cv=kfold, scoring="accuracy", n_jobs=-1
    )

    return scores.mean()


study = optuna.create_study(direction="maximize")
study.optimize(objective_forest, n_trials=50)

print("Meilleur score Optuna :", study.best_value, "\n")
print("Meilleurs hyperparamètres :")
print(study.best_params)

In [ ]:
df["science_related"].value_counts(normalize=True)

# 1. Séparation en 2 DataFrames (votre idée)
df_0 = df[df["science_related"] == 0]
df_1 = df[df["science_related"] == 1]

# 2. On récupère le nombre de lignes de la classe minoritaire (les 1)
nb_lignes_1 = len(df_1)

# 3. On pioche au hasard le même nombre de lignes dans les 0
# random_state=42 permet de bloquer le hasard pour avoir toujours le même résultat
df_0_sous_echantillonne = df_0.sample(n=nb_lignes_1, random_state=42)

# 4. On fusionne les deux morceaux dans un nouveau DataFrame
df_equilibre = pd.concat([df_1, df_0_sous_echantillonne])

# 5. Optionnel : On mélange les lignes pour ne pas avoir tous les 1 puis tous les 0
df_equilibre = df_equilibre.sample(frac=1, random_state=42).reset_index(drop=True)

# Vérification
print(df_equilibre["science_related"].value_counts())

# Science Related

In [ ]:
params = {
    "model__C": [0.01, 0.1, 1, 5, 10],
    "model__penalty": ["l2"],        # LinearSVC: l1 requires loss='squared_hinge' + dual=False
    "model__loss": ["hinge", "squared_hinge"],
    "model__max_iter": [5000, 10000, 20000], # avoid convergence warnings
    "vectorizer__ngram_range": [(1,1), (1,2), (1,3)],
    "vectorizer__max_features": [3000, 5000, 10000],
    "vectorizer__min_df": [2, 5, 10],
    "vectorizer__max_df": [0.85, 0.95],
}

# If you also want l1, add it as a separate param grid (list of dicts)
params = [
    {   # l2 supports both loss types
        "model__C": [0.01, 0.1, 1, 5, 10],
        "model__penalty": ["l2"],
        "model__loss": ["hinge", "squared_hinge"],
        "model__max_iter": [5000, 10000, 20000],
        "vectorizer__ngram_range": [(1,1), (1,2), (1,3)],
        "vectorizer__max_features": [3000, 5000, 10000],
        "vectorizer__min_df": [2, 5, 10],
        "vectorizer__max_df": [0.85, 0.95],
    },
    {   # l1 only works with squared_hinge + dual=False
        "model__C": [0.01, 0.1, 1, 5, 10],
        "model__penalty": ["l1"],
        "model__loss": ["squared_hinge"],
        "model__dual": [False],
        "model__max_iter": [5000, 10000, 20000],
        "vectorizer__ngram_range": [(1,1), (1,2), (1,3)],
        "vectorizer__max_features": [3000, 5000, 10000],
        "vectorizer__min_df": [2, 5, 10],
        "vectorizer__max_df": [0.85, 0.95],
    }
]

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
grid_param_rf = {
    "n_estimators": [4, 6, 9],
    "max_features": ["log2", "sqrt"],
    "criterion": ["entropy", "gini"],
    "max_depth": [2, 3, 5, 10],
    "min_samples_split": [2, 3, 5],
    "min_samples_leaf": [1, 5, 8]
}
gd_sr_rf = GridSearchCV(
    estimator=RandomForestClassifier(),
    param_grid=grid_param_rf,
    scoring="accuracy",
    cv=5,
    n_jobs=-1,
    return_train_score=True
)

gd_sr_rf.fit(X_train_related, y_train_related)

print("Meilleur score :", gd_sr_rf.bestscore, "\n")
print("Meilleurs paramètres :", gd_sr_rf.bestparams, "\n")
print("Meilleur estimateur :", gd_sr_rf.bestestimator, "\n")

# --- Overfitting diagnosis ---
results = pd.DataFrame(grid.cv_results_)
train_score = results.loc[grid.best_index_, "mean_train_score"]
val_score   = grid.best_score_
gap         = train_score - val_score

print("Meilleurs paramètres :", grid.best_params_)
print(f"Train score:      {train_score:.4f}")
print(f"Validation score: {val_score:.4f}")
print(f"Overfitting gap:  {gap:.4f}", "⚠️ possible overfit" if gap > 0.1 else "✅ looks healthy")

# --- Final evaluation ---
test_score = grid.score(X_test_related, y_test_related)
print(f"\nTest score (held-out): {test_score:.4f}")
print(classification_report(y_test_related, grid.predict(X_test_related)))

# --- Saving ---
model_path = "ScientificModel.pkl"
with open(model_path, "wb") as f:
    pickle.dump(grid.best_estimator_, f)
print("Modèle sauvegardé dans :", model_path)

# Génération et test du model ((claim,ref) vs context)

In [ ]:
# --- Préparation des données ---

# Filtrer uniquement les lignes science_related
df_sci = df.dropna(subset=["scientific_claim", "scientific_reference", "scientific_context"])

# Les 2 colonnes cibles — supposées binaires (0/1)
df_sci = df_sci.copy()
df_sci["claim_or_ref"] = ((df_sci["scientific_claim"] == 1) | (df_sci["scientific_reference"] == 1)).astype(int)
df_sci["context"]      = (df_sci["scientific_context"] == 1).astype(int)
TARGET_COLS = ["claim_or_ref", "context"]
LABEL_NAMES = ["CLAIM/REF", "CONTEXT"]

# --- Split ---
texte = [preprocess(t) for t in df_sci.text]
Y = df_sci[TARGET_COLS].values  # shape (n_samples, 3)
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size="0.2", random_state=42)
train_idx, test_idx = next(msss.split(texte, Y))
X_train_double = [texte[i] for i in train_idx]
X_test_double  = [texte[i] for i in test_idx]
y_train_double = Y[train_idx]
y_test_double  = Y[test_idx]

# --- Pipeline avec OneVsRestClassifier ---
pipe = Pipeline([
    ("vectorizer", TfidfVectorizer(lowercase=False, ngram_range=(1, 2))),
    ("model", OneVsRestClassifier(LinearSVC(), n_jobs=-1))
])

# --- Grille de paramètres ---
params = [
    {
        "model__estimator__C": [0.01, 0.1, 1, 5, 10],
        "model__estimator__penalty": ["l2"],
        "model__estimator__loss": ["hinge", "squared_hinge"],
        "model__estimator__max_iter": [5000, 10000, 20000],
        "vectorizer__ngram_range": [(1,1), (1,2), (1,3)],
        "vectorizer__max_features": [3000, 5000, 10000],
        "vectorizer__min_df": [2, 5, 10],
        "vectorizer__max_df": [0.85, 0.95],
    },
    {
        "model__estimator__C": [0.01, 0.1, 1, 5, 10],
        "model__estimator__penalty": ["l1"],
        "model__estimator__loss": ["squared_hinge"],
        "model__estimator__dual": [False],
        "model__estimator__max_iter": [5000, 10000, 20000],
        "vectorizer__ngram_range": [(1,1), (1,2), (1,3)],
        "vectorizer__max_features": [3000, 5000, 10000],
        "vectorizer__min_df": [2, 5, 10],
        "vectorizer__max_df": [0.85, 0.95],
    }
]

# --- GridSearch ---
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
cv = MultilabelStratifiedKFold(n_splits=10, shuffle=True, random_state=42)
grid = GridSearchCV(
    pipe, params,
    cv=cv,
    scoring="f1_weighted",
    return_train_score=True,
    n_jobs=-1
)


warnings.filterwarnings("ignore", category=ConvergenceWarning)
grid.fit(X_train_double, y_train_double)
warnings.filterwarnings("default", category=ConvergenceWarning)

# --- Diagnostic overfitting ---
results = pd.DataFrame(grid.cv_results_)
train_score = results.loc[grid.best_index_, "mean_train_score"]
val_score   = grid.best_score_
gap         = train_score - val_score

print("Meilleurs paramètres :", grid.best_params_)
print(f"Train score:      {train_score:.4f}")
print(f"Validation score: {val_score:.4f}")
print(f"Overfitting gap:  {gap:.4f}", "⚠️ possible overfit" if gap > 0.1 else "✅ looks healthy")

# --- Évaluation par classe ---
y_pred = grid.predict(X_test_double)  # shape (n_samples, 3)

for i, label in enumerate(LABEL_NAMES):
    print(f"\n--- Classe : {label} ---")
    print(classification_report(y_test_double[:, i], y_pred[:, i]))

# --- Prédiction : 3 sorties séparées ---
def predict_all(texts, model):
    """Retourne un DataFrame avec une colonne par classe."""
    preds = model.predict(texts)  # shape (n, 3)
    return pd.DataFrame(preds, columns=LABEL_NAMES)

# Exemple sur le test set
predictions_df = predict_all(X_test_double, grid.best_estimator_)
print(predictions_df.head(10))

# --- Sauvegarde ---
model_path = "ScientificTypeBinModel.pkl"
with open(model_path, "wb") as f:
    pickle.dump(grid.best_estimator_, f)
print("Modèle sauvegardé dans :", model_path)

# Génération et test du model (claim vs ref vs context)

In [ ]:
# --- Préparation des données ---

# Filtrer uniquement les lignes science_related
df_sci = df.dropna(subset=["scientific_claim", "scientific_reference", "scientific_context"])

# Les 3 colonnes cibles — supposées binaires (0/1)
TARGET_COLS = ["scientific_claim", "scientific_reference", "scientific_context"]
LABEL_NAMES = ["CLAIM", "REF", "CONTEXT"]

# --- Split ---
texte = [preprocess(t) for t in df_sci.text]
Y = df_sci[TARGET_COLS].values  # shape (n_samples, 3)
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(msss.split(texte, Y))
X_train_triple = [texte[i] for i in train_idx]
X_test_triple  = [texte[i] for i in test_idx]
y_train_triple = Y[train_idx]
y_test_triple  = Y[test_idx]

# --- Pipeline avec OneVsRestClassifier ---
pipe = Pipeline([
    ("vectorizer", TfidfVectorizer(lowercase=False, ngram_range=(1, 2))),
    ("model", OneVsRestClassifier(LinearSVC(), n_jobs=-1))
])

# --- Grille de paramètres ---
params = [
    {
        "model__estimator__C": [0.01, 0.1, 1, 5, 10],
        "model__estimator__penalty": ["l2"],
        "model__estimator__loss": ["hinge", "squared_hinge"],
        "model__estimator__max_iter": [5000, 10000, 20000],
        "vectorizer__ngram_range": [(1,1), (1,2), (1,3)],
        "vectorizer__max_features": [3000, 5000, 10000],
        "vectorizer__min_df": [2, 5, 10],
        "vectorizer__max_df": [0.85, 0.95],
    },
    {
        "model__estimator__C": [0.01, 0.1, 1, 5, 10],
        "model__estimator__penalty": ["l1"],
        "model__estimator__loss": ["squared_hinge"],
        "model__estimator__dual": [False],
        "model__estimator__max_iter": [5000, 10000, 20000],
        "vectorizer__ngram_range": [(1,1), (1,2), (1,3)],
        "vectorizer__max_features": [3000, 5000, 10000],
        "vectorizer__min_df": [2, 5, 10],
        "vectorizer__max_df": [0.85, 0.95],
    }
]

# --- GridSearch ---
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
cv = MultilabelStratifiedKFold(n_splits=10, shuffle=True, random_state=42)
grid = GridSearchCV(
    pipe, params,
    cv=cv,
    scoring="f1_weighted",
    return_train_score=True,
    n_jobs=-1
)


warnings.filterwarnings("ignore", category=ConvergenceWarning)
grid.fit(X_train_triple, y_train_triple)
warnings.filterwarnings("default", category=ConvergenceWarning)

# --- Diagnostic overfitting ---
results = pd.DataFrame(grid.cv_results_)
train_score = results.loc[grid.best_index_, "mean_train_score"]
val_score   = grid.best_score_
gap         = train_score - val_score

print("Meilleurs paramètres :", grid.best_params_)
print(f"Train score:      {train_score:.4f}")
print(f"Validation score: {val_score:.4f}")
print(f"Overfitting gap:  {gap:.4f}", "⚠️ possible overfit" if gap > 0.1 else "✅ looks healthy")

# --- Évaluation par classe ---
y_pred = grid.predict(X_test_triple)  # shape (n_samples, 3)

for i, label in enumerate(LABEL_NAMES):
    print(f"\n--- Classe : {label} ---")
    print(classification_report(y_test_triple[:, i], y_pred[:, i]))

# --- Prédiction : 3 sorties séparées ---
def predict_all(texts, model):
    """Retourne un DataFrame avec une colonne par classe."""
    preds = model.predict(texts)  # shape (n, 3)
    return pd.DataFrame(preds, columns=LABEL_NAMES)

# Exemple sur le test set
predictions_df = predict_all(X_test_triple, grid.best_estimator_)
print(predictions_df.head(10))

# --- Sauvegarde ---
model_path = "ScientificTypeModel.pkl"
with open(model_path, "wb") as f:
    pickle.dump(grid.best_estimator_, f)
print("Modèle sauvegardé dans :", model_path)

# Loading et requête au model (Scientific vs non scientific)

In [ ]:
# --- Loading ---
model_path = "ScientificModel.pkl"
scientific_model = pickle.load(open(model_path, "rb"))

# Loading et requête au model (claim vs ref vs context)

In [ ]:
# --- Loading ---
model_path = "ScientificTypeBinModel.pkl"
claim_ref_context_bin_model = pickle.load(open(model_path, "rb"))

# Loading et requête au model (claim vs ref vs context)

In [ ]:
# --- Loading ---
model_path = "ScientificTypeModel.pkl"
claim_ref_context_model = pickle.load(open(model_path, "rb"))

# Prédiction wowowowowowowow

In [ ]:

review = "study shows that studying math increase fertility"
preprocessed_review = preprocess(review)
scientific_prediction = scientific_model.predict([preprocessed_review])[0]
claim_ref_context_prediction = claim_ref_context_model.predict([preprocessed_review])[0]
print("Avis :", review)
print("Avis préprocess :", preprocessed_review)
print("Scientific Prediction :", scientific_prediction)
print("Claim vs REF vs CONTEXT Prediction :", claim_ref_context_prediction, [("" if col == 1 else "NOT ") + ["CLAIM", "REF", "CONTEXT"][i] for i,col in enumerate(claim_ref_context_prediction)])